In [30]:
from gliner import GLiNER
from gliner import GLiNER
import os
os.getcwd()

import pandas as pd
import numpy as np
from collections import defaultdict
from tqdm import tqdm
tqdm.pandas()

In [15]:
os.getcwd()

'/teamspace/studios/this_studio/diss_git'

In [2]:
model = GLiNER.from_pretrained("urchade/gliner_medium-v2.1")

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

/home/zeus/miniconda3/envs/cloudspace/lib/python3.10/site-packages/transformers/convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


In [24]:
def extract_gliner_test(input_text, labels=labels, confidence_threshold=0.3):

    # initiating dictionary to store extracted entities
    entities_dict = defaultdict(list)

    # extracting entities using GLINER
    entities = model.predict_entities(input_text, labels, threshold=confidence_threshold)
    
    # grouping entities by label and adding to dictionary
    for entity in entities:
        entities_dict[entity['label']].append(entity['text'])

    for label in labels:
        if label in entities_dict and entities_dict[label]:
            # Remove duplicates while preserving case-insensitive uniqueness
            # Keep the first occurrence of each unique entity (case-insensitive)
            unique_entities = []
            seen_lower = set()
            
            for entity in entities_dict[label]:
                entity_lower = entity.lower()
                if entity_lower not in seen_lower:
                    unique_entities.append(entity)
                    seen_lower.add(entity_lower)
            
            # Join the list of unique entities into a single, comma-separated string
            formatted_list = ", ".join(unique_entities)
            print(f"{label}: [{formatted_list}]")
        else:
            print(f"{label}: No entities found")

In [25]:
# FUNCTION FOR LOOPING OVER MULTIPLE ROWS AND RETURNING DF

def extract_gliner(input_text, labels=labels, confidence_threshold=0.3):

    # initiating dictionary to store extracted entities
    entities_dict = defaultdict(list)

    # extracting entities using GLINER
    entities = model.predict_entities(input_text, labels, threshold=confidence_threshold)
    
    # grouping entities by label and adding to dictionary
    for entity in entities:
        entities_dict[entity['label']].append(entity['text'])

    result = {}
    for label in labels:
        if entities_dict[label]: # if the label is not empty
            # Remove duplicates while preserving case-insensitive uniqueness
            # Keep the first occurrence of each unique entity (case-insensitive)
            unique_entities = []
            seen_lower = set()
            
            for entity in entities_dict[label]:
                entity_lower = entity.lower()
                if entity_lower not in seen_lower:
                    unique_entities.append(entity)
                    seen_lower.add(entity_lower) # seen_lower is then banished to the void

            result[f"extracted_{label}"] = ",".join(unique_entities)
        else: # if the label is empty
            result[f"extracted_{label}"] = None

    return pd.Series(result)

In [19]:
labels = ["event", "location"]

text = """
STALYBRIDGE.—A public meeting was held in the People’s School here on Monday evening last, when the National Petition was read and adopted; after which, Mr. James Leach, of Manchester, delivered an address, exposing the fallacies of the Corn Law repealers. A Corn Law lecture had been previously delivered in the town, by a Mr. Spencer, to about half a dozen of the middle classes; the Chartists, however, upset his meeting.  WOOLWICH.—STRIKE OF THE MASONS.—A public meeting of the inhabitants of Woolwich was held on Thursday evening, Oct. 28th, in the theatre of that town, for the purpose of laying before the inhabitants every particular connected with the strike of the masons at the New Houses of Parliament, Nelson’s Monument, and Woolwich Dock Yard, also, to take into consideration the conduct of a portion of the metropolitan press. The meeting was called for seven o’clock, and long before that hour, the theatre was thronged in every part, the boxes being filled with well-dressed females. Mr. Maddox was called to the chair, and the meeting was addressed by Mr. Davies, Mr. Carter, Mr. Wood, Mr. Parker, Mr. Walton, Dr. McDouall, Captain Aokerley, and others. The meeting consisted of about a thousand persons. We are obliged to the kindness of a friend for furnishing us with a long report of this meeting, a favour which would have been greatly enhanced had it reached us before Thursday morning last; just one week after the meeting had been held, and too late to be made use of at length for the Star.
"""
extract_gliner(text, labels=labels)

event: [public meeting, National Petition, Corn Law lecture, STRIKE OF THE MASONS]
location: [STALYBRIDGE, People’s School, Manchester, WOOLWICH, New Houses of Parliament, Nelson’s Monument, theatre]


In [17]:
simple_test = """
Weavers and card-room hands, attend the meeting which will be held in the Charlestown meeting room, on Wednesday evening, Nov. 6th, at eight o'clock, and show by your thousands that you are determined to be no longer 'stumped upon with impunity.
"""

labels = ["event", "location"]
extract_gliner(simple_test, labels=labels, confidence_threshold=0.39)

event: [Wednesday evening]
location: [Charlestown meeting room]


In [6]:
test2 = """
EMERR'S BRIGADE.—A public meeting, in favour of the People's Charter, will be held at the Bricklayers' Arms, Homer-street, New Road, Mary-lebone, on Monday evening next, the 18th inst., at eight o'clock precisely. Messrs. Mantz and Davoé will attend.  MR. SKELTON will deliver a lecture at the Standard of Liberty, Brick-lane, Spitalfields, on Sunday evening next, the 17th inst., at half-past seven precisely.  SOMERS' TOWN LOCALITY.—On Sunday evening next, Mr. Mee will lecture at Mr. Dudbridges, Bricklayers' Arms, Tonbridge-street, New Road.  MR. HUNNIBALL, of Stafford, will deliver a lecture on Sunday, the 17th inst., at the Golden Lion, Dean-street, Soho, on the causes of the Revolutions of Greece and Rome.  LONDON DISTRICT COUNCIL.—This Council will meet at the City of London Political and Scientific Institution, Turnagain Lane, on Sunday afternoon next, the 17th inst., at three o'clock precisely.  CAMBERWELL.—A public meeting will be held at the Cook Tavern, Camberwell Green, on Tuesday next, the 19th inst., at eight precisely.  HAMMERSMITH, NOTTINGHILL, AND THEIR VICINITIES.—The Chartists and their friends of the above places are most respectfully requested to attend a meeting at the Black Bull Inn, Hammersmith Road, on Tuesday evening next, the 19th inst., at eight precisely, on business of great importance."""

labels = ["event", "location"]
extract_gliner(test2, labels=labels, confidence_threshold=0.3)

event: [Revolutions of Greece and Rome, public meeting]
location: [Bricklayers' Arms, Homer-street, New Road, Mary-lebone, Standard of Liberty, Brick-lane, Spitalfields, SOMERS' TOWN LOCALITY, Tonbridge-street, Stafford, Golden Lion, Dean-street, Soho, City of London Political and Scientific Institution, Turnagain Lane, CAMBERWELL, Cook Tavern, Camberwell Green, HAMMERSMITH, NOTTINGHILL, Black Bull Inn, Hammersmith Road]


In [16]:
checked_samples = pd.read_csv("./data/subset_samples_checked.csv")
checked_samples.info()

In [38]:
# as a test, running extract_gliner on the first 20 rows of the 'article' column in the checked samples df

extracted_data = checked_samples['article'].head(20).progress_apply(
    lambda x: extract_gliner(x, 
                             labels=labels,
                             confidence_threshold=0.4)
)

# creating new df, adding extracted entities to select rows of checked samples df
samples_extracted = pd.concat([
    checked_samples[['corpus_id', 'score', 'date', 'source', 'article']].head(20),
    extracted_data], axis=1)

# viewing new df
samples_extracted


100%|██████████| 20/20 [00:14<00:00,  1.42it/s]


,corpus_id,score,date,source,article,extracted_event,extracted_location
0,526262,0.749631,1843-03-11,star,"is concerned, we had no complaint to make. Na...",None,"Bridgewater works,Worsley,Eccles,Manchester"
1,895228,0.744991,1841-11-06,star,STALYBRIDGE.—A public meeting was held in the ...,"National Petition,Corn Law lecture,STRIKE OF T...","STALYBRIDGE,Manchester,WOOLWICH,New Houses of ..."
2,625534,0.742783,1845-05-31,star,"Mr. S. Crawfurn presented a petition, signed b...","Field Gardens Bill,Ten Hours' Bill","Liverpool,London,Edin- burgh,Kettering,Notting..."
3,391392,0.741237,1843-03-18,star,"see no cause for complaint, or, at least, no c...",Monday evening,"Freemasons' Tavern,Fleet-street"
4,817816,0.720612,1842-10-01,star,A public conversational meeting was held on Th...,None,"Ship Inn,Long Lane,Bermondsey"
5,746042,0.712609,1845-08-23,star,"On Thursday I proceeded to Burnley, and was me...",None,"Burnley,Blackburn,Padiham,Temperance Hotel,Colne"
6,38962,0.704015,1842-07-30,star,COVENTRY.—The cause goes on well here. We have...,"Lectures,Sales","COVENTRY,Foleshill,Bulkington,Kenilworth"
7,308112,0.701977,1849-03-24,star,TO THE EDITOR OF THE NORTHERN STAR. SIR—As th...,"Charter,deputation meeting","Gloucester,Worcester,Cheltenham,Tewkesbury,New..."
8,639552,0.700175,1842-01-15,star,We must do all we can to assist “the old King”...,None,"asylum,Parliament"
9,306304,0.699766,1850-09-14,star,"last forty years, making the manufacturers ric...",None,"kingdom,London"


In [39]:
samples_extracted.to_csv("./data/extractedentities_test.csv")